# step_renderer

> Phase 2 step renderer: Structural Decomposition with card stack UI and keyboard navigation

In [ ]:
#| default_exp components.step_decomposition.step_renderer

In [ ]:
#| export
from typing import Any, List

from fasthtml.common import Div, H2, P, Button, Span
from cjm_fasthtml_interactions.core.context import InteractionContext

# DaisyUI components
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_sizes, btn_styles, btn_colors
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, h
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, justify, items, gap, grow
)
from cjm_fasthtml_tailwind.core.base import combine_classes

# Lucide icons
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Keyboard navigation
from cjm_fasthtml_keyboard_navigation.components.system import render_keyboard_system

# Card stack library
from cjm_fasthtml_card_stack.components.viewport import render_viewport
from cjm_fasthtml_card_stack.components.controls import (
    render_width_slider, render_card_count_select
)
from cjm_fasthtml_card_stack.components.progress import render_progress_indicator
from cjm_fasthtml_card_stack.components.states import render_loading_state
from cjm_fasthtml_card_stack.core.models import CardStackState
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT, DEFAULT_CARD_WIDTH
from cjm_fasthtml_card_stack.keyboard.actions import (
    build_card_stack_url_map, render_card_stack_action_buttons
)

# Token selector library
from cjm_fasthtml_token_selector.js.core import generate_token_selector_js
from cjm_fasthtml_token_selector.components.inputs import render_hidden_inputs as render_ts_hidden_inputs

# Card stack + token selector configuration (Phase 2 decomposition instance)
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.card_stack_config import (
    DECOMP_CS_CONFIG, DECOMP_CS_IDS, DECOMP_CS_BTN_IDS,
    DECOMP_TS_CONFIG, DECOMP_TS_IDS,
)

# Local imports
from cjm_fasthtml_workflow_transcript_decomp.core.html_ids import StructureDecompHtmlIds
from cjm_fasthtml_workflow_transcript_decomp.core.models import WorkingSegment
from cjm_fasthtml_workflow_transcript_decomp.routes.models import DecompUrls
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.helpers import (
    _get_decomposition_state, _get_segments, _get_focused_index, _is_initialized, _get_history,
    _get_visible_count, _get_card_width,
)
from cjm_fasthtml_workflow_transcript_decomp.services.text_utils import calculate_segment_stats
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.segment_card import (
    create_segment_card_renderer
)
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.callbacks import (
    generate_decomp_callbacks_script
)
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.keyboard_config import (
    SD_DECOMP_ENTER_SPLIT_BTN, SD_DECOMP_EXIT_SPLIT_BTN,
    SD_DECOMP_SPLIT_BTN, SD_DECOMP_MERGE_BTN, SD_DECOMP_UNDO_BTN,
    _create_decomposition_keyboard_manager, _render_decomposition_keyboard_hints,
)

## Decomposition Toolbar & Stats

Workflow-specific toolbar with undo/reset/AI split buttons and segment statistics.
These are rendered alongside the generic card stack viewport.

In [ ]:
#| export
def render_toolbar(
    reset_url: str,  # URL for reset action
    ai_split_url: str,  # URL for AI split action
    undo_url: str,  # URL for undo action
    can_undo: bool,  # Whether undo is available
    visible_count: int = DEFAULT_VISIBLE_COUNT,  # Current visible card count
    oob: bool = False,  # Whether to render as OOB swap
) -> Any:  # Toolbar component
    """Render the decomposition toolbar with action buttons and card count selector."""
    return Div(
        # Left group: Undo button
        # Note: no id= here — the keyboard system owns the "sd-decomp-undo-btn" ID
        # for its hidden action button. This visible button works via hx_post directly.
        Button(
            lucide_icon("undo-2", size=4, cls=str(m.r(2))),
            "Undo",
            cls=combine_classes(btn, btn_styles.ghost, btn_sizes.sm),
            disabled=None if can_undo else "disabled",
            hx_post=undo_url,
            hx_swap="none"
        ),

        # Left spacer
        Div(cls=str(grow())),

        # Center: Card count selector with label
        Div(
            Span(
                "Cards:",
                cls=combine_classes(font_size.sm, text_dui.base_content.opacity(70), m.r(2))
            ),
            render_card_count_select(DECOMP_CS_CONFIG, DECOMP_CS_IDS, current_count=visible_count),
            cls=combine_classes(flex_display, items.center)
        ),

        # Right spacer
        Div(cls=str(grow())),

        # Right group: Reset and AI Split buttons
        Div(
            Button(
                lucide_icon("rotate-ccw", size=4, cls=str(m.r(2))),
                "Reset",
                id=StructureDecompHtmlIds.DECOMP_RESET_BTN,
                cls=combine_classes(btn, btn_styles.ghost, btn_sizes.sm),
                hx_post=reset_url,
                hx_swap="none"
            ),
            Button(
                lucide_icon("sparkles", size=4, cls=str(m.r(2))),
                "AI Split",
                id=StructureDecompHtmlIds.DECOMP_AI_SPLIT_BTN,
                cls=combine_classes(btn, btn_colors.secondary, btn_sizes.sm),
                hx_post=ai_split_url,
                hx_swap="none"
            ),
            cls=combine_classes(flex_display, gap(2))
        ),

        id=StructureDecompHtmlIds.DECOMP_TOOLBAR,
        cls=combine_classes(
            flex_display, gap(2), items.center
        ),
        hx_swap_oob="true" if oob else None
    )

In [ ]:
#| export
def render_decomp_stats(
    segments: List[WorkingSegment],  # Current segments
    oob: bool = False,  # Whether to render as OOB swap
) -> Any:  # Statistics component
    """Render decomposition statistics."""
    stats = calculate_segment_stats(segments)

    return Div(
        Span(
            f"{stats['total_segments']} segments · {stats['total_words']:,} words",
            cls=combine_classes(font_size.sm, text_dui.base_content.opacity(70))
        ),
        id=StructureDecompHtmlIds.DECOMP_STATS,
        hx_swap_oob="true" if oob else None
    )

## Shared Content Renderer

Renders the full decomposition step content including keyboard system.
Called by both `render_decomposition_step` (when initialized) and the init route handler.

In [ ]:
#| export
def _render_decomposition_content(
    segments:List[WorkingSegment],  # Segments to display
    focused_index:int,  # Currently focused segment index
    history_depth:int,  # Undo history depth (for button state)
    visible_count:int,  # Number of visible cards in viewport
    card_width:int,  # Card stack width in rem
    urls:DecompUrls,  # URL bundle for all decomposition routes
) -> Any:  # FastHTML component with full decomposition UI
    """Render complete decomposition step content with keyboard navigation."""
    total_segments = len(segments)

    # Create keyboard manager using library focus zone + nav actions + decomp-specific actions
    kb_manager = _create_decomposition_keyboard_manager(
        ids=DECOMP_CS_IDS,
        button_ids=DECOMP_CS_BTN_IDS,
        config=DECOMP_CS_CONFIG,
    )

    # Include selector: library's focused index input + token selector anchor input
    include_selector = (
        f"#{DECOMP_CS_IDS.focused_index_input}, "
        f"#{DECOMP_TS_IDS.anchor_input}"
    )

    # URL map: library nav buttons + consumer decomp buttons
    url_map = {
        **build_card_stack_url_map(DECOMP_CS_BTN_IDS, urls.card_stack),
        SD_DECOMP_ENTER_SPLIT_BTN: urls.enter_split,
        SD_DECOMP_EXIT_SPLIT_BTN: urls.exit_split,
        SD_DECOMP_SPLIT_BTN: urls.split,
        SD_DECOMP_MERGE_BTN: urls.merge,
        SD_DECOMP_UNDO_BTN: urls.undo,
    }

    # All actions target the card stack with swap=none (OOB updates only)
    card_stack_target = f"#{DECOMP_CS_IDS.card_stack}"
    target_map = {btn_id: card_stack_target for btn_id in url_map}
    include_map = {btn_id: include_selector for btn_id in url_map}
    swap_map = {btn_id: "none" for btn_id in url_map}

    kb_system = render_keyboard_system(
        kb_manager,
        url_map=url_map,
        target_map=target_map,
        include_map=include_map,
        swap_map=swap_map,
        show_hints=False,
        include_state_inputs=True,
    )

    # Create card renderer callback
    card_renderer = create_segment_card_renderer(
        split_url=urls.split,
        merge_url=urls.merge,
        enter_split_url=urls.enter_split,
        exit_split_url=urls.exit_split,
    )

    # Build CardStackState for library viewport
    cs_state = CardStackState(
        focused_index=focused_index,
        visible_count=visible_count,
        card_width=card_width,
    )

    # Render viewport from library (includes focused index hidden input)
    viewport = render_viewport(
        card_items=segments,
        state=cs_state,
        config=DECOMP_CS_CONFIG,
        ids=DECOMP_CS_IDS,
        urls=urls.card_stack,
        render_card=card_renderer,
        form_input_name="segment_index",
    )

    # Generate JS: library card stack JS + focus change callback
    callbacks_script = generate_decomp_callbacks_script(
        ids=DECOMP_CS_IDS,
        button_ids=DECOMP_CS_BTN_IDS,
        config=DECOMP_CS_CONFIG,
        urls=urls.card_stack,
        container_id=StructureDecompHtmlIds.DECOMP_CONTAINER,
        focus_input_id=DECOMP_CS_IDS.focused_index_input,
    )

    # Token selector JS (caret navigation, display, key repeat)
    ts_script = generate_token_selector_js(DECOMP_TS_CONFIG, DECOMP_TS_IDS)

    return Div(
        # Header with title, description, keyboard hints, toolbar, and width slider
        Div(
            H2("Decompose Structure", cls=combine_classes(font_size._3xl, font_weight.bold)),
            P(
                "Split and merge text segments to create the document structure.",
                cls=combine_classes(text_dui.base_content.opacity(70), m.b(4))
            ),
            _render_decomposition_keyboard_hints(kb_manager),
            # Toolbar with card count selector
            Div(
                render_toolbar(
                    reset_url=urls.reset,
                    ai_split_url=urls.ai_split,
                    undo_url=urls.undo,
                    can_undo=(history_depth > 0),
                    visible_count=visible_count,
                ),
                cls=combine_classes(m.t(4))
            ),
            # Width slider below toolbar
            Div(
                render_width_slider(DECOMP_CS_CONFIG, DECOMP_CS_IDS, card_width=card_width),
                cls=combine_classes(m.t(4))
            ),
        ),

        # Segment viewport
        viewport,

        # Footer with progress and stats
        Div(
            render_progress_indicator(focused_index, total_segments, DECOMP_CS_IDS, label="Segment"),
            render_decomp_stats(segments),
            id=StructureDecompHtmlIds.DECOMP_FOOTER,
            cls=combine_classes(
                p(4), bg_dui.base_100,
                flex_display, justify.between, items.center, gap(4)
            )
        ),

        # Keyboard navigation system
        kb_system.script,
        kb_system.hidden_inputs,
        kb_system.action_buttons,

        # Hidden action buttons for JS-triggered card stack nav (page_up, page_down, first, last)
        render_card_stack_action_buttons(DECOMP_CS_BTN_IDS, urls.card_stack, DECOMP_CS_IDS),

        # Token selector hidden inputs (anchor + focus)
        *render_ts_hidden_inputs(DECOMP_TS_IDS),

        # JavaScript callbacks (library card stack JS + focus change)
        callbacks_script,

        # Token selector JS (caret navigation, display, key repeat)
        ts_script,

        id=StructureDecompHtmlIds.DECOMP_CONTAINER,
        cls=combine_classes(
            w.full, h.full,
            flex_display, flex_direction.col,
            p(4)
        )
    )

## Main Step Renderer

In [ ]:
#| export
def render_decomposition_step(
    ctx: InteractionContext,  # Interaction context with state and data
    urls: DecompUrls = None,  # URL bundle for decomposition routes
) -> Any:  # FastHTML component
    """Render Phase 2: Structural Decomposition step with segment viewport UI."""
    urls = urls or DecompUrls()

    # Check if segments are initialized
    is_init = _is_initialized(ctx)

    # If not initialized, show loading state with auto-trigger for init
    if not is_init:
        return Div(
            # Header
            Div(
                H2("Decompose Structure", cls=combine_classes(font_size._3xl, font_weight.bold)),
                P(
                    "Split and merge text segments to create the document structure.",
                    cls=combine_classes(text_dui.base_content.opacity(70), m.b(4))
                ),
                cls=combine_classes(m.b(4))
            ),

            # Loading state from card stack library
            render_loading_state(DECOMP_CS_IDS, message="Initializing segments..."),

            # Auto-trigger initialization
            Div(
                hx_post=urls.init,
                hx_trigger="load",
                hx_target=StructureDecompHtmlIds.as_selector(StructureDecompHtmlIds.DECOMP_CONTAINER),
                hx_swap="outerHTML"
            ) if urls.init else None,

            id=StructureDecompHtmlIds.DECOMP_CONTAINER,
            cls=combine_classes(
                w.full, h.full,
                flex_display, flex_direction.col,
                p(4)
            )
        )

    # Get data from state
    segments = _get_segments(ctx)
    focused_index = _get_focused_index(ctx)
    history = _get_history(ctx)
    visible_count = _get_visible_count(ctx, default=DEFAULT_VISIBLE_COUNT)
    card_width = _get_card_width(ctx, default=DEFAULT_CARD_WIDTH)

    # Use shared content renderer
    return _render_decomposition_content(
        segments=segments,
        focused_index=focused_index,
        history_depth=len(history),
        visible_count=visible_count,
        card_width=card_width,
        urls=urls,
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()